# Наивный Байес для смешанных признаков

Доход — то поле, которое люди отказываются заполнять. Возраст, образование, сферу занятости,
отработанные часы они обычно назовут. Поэтому вопрос, который стоит денег, звучит так: можно ли по
имеющимся полям узнать то, которого нет.

Датасет этого ноутбука ставит ровно такой вопрос на записях переписи населения США 1994 года:
заработал ли этот человек больше \$50K? Каждую запись описывают четырнадцать полей, и они двух
сортов — метки вроде сферы занятости и числа вроде возраста. Ваш классификатор должен читать и те,
и другие.

Работа — четыре пропуска в **`naive_bayes.py`**, и только этот модуль оценивается. Разделы идут в
том порядке, в котором стоит работать: сначала посмотреть на данные, дать им подсказать, какие
бывают виды столбцов, и выписать модель уже тогда, когда известно, с чем ей иметь дело.

In [ ]:
# На Colab: качаем вспомогательные модули и заготовку, лежащие рядом с ноутбуком в репозитории.
import pathlib
import urllib.request

BASE_URL = (
    "https://raw.githubusercontent.com/girafe-ai/ml-course/homeworks/contest/classic_naive_bayes"
)

try:
    import ucimlrepo
except ImportError:
    %pip install --quiet ucimlrepo

for module in ("mixed_nb_data.py", "mixed_nb_eval.py", "naive_bayes.py"):
    if not pathlib.Path(module).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{module}", module)
        print(f"скачан {module}")

In [ ]:
import importlib
import time

import matplotlib.pyplot as plt
import mixed_nb_data
import mixed_nb_eval
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import CategoricalNB, GaussianNB
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from ucimlrepo import fetch_ucirepo

pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

## 1. Датасет

[Adult](https://archive.ics.uci.edu/dataset/2/adult) — это и есть та выгрузка переписи: около
49 000 респондентов, по четырнадцать полей на каждого и метка \$50K. `ucimlrepo` скачивает его и —
что для дальнейшего даже полезнее — отдаёт собственные *метаданные переменных* UCI: небольшую
таблицу, которая называет каждый столбец, говорит, признак это или цель, и объявляет его тип.

Держите эту таблицу под рукой. Раздел 4 на неё опирается, а раздел 2 отчасти про то, где она
расходится с тем, как выглядят значения.

In [ ]:
adult = fetch_ucirepo(id=2)

TARGET = str(adult.data.targets.columns[0])
data = pd.concat([adult.data.features, adult.data.targets], axis=1)
variables = adult.variables.set_index("name")

print(f"{len(data)} строк, {data.shape[1] - 1} признаков, целевая переменная {TARGET!r}")
display(variables.loc[list(data.columns), ["role", "type", "description"]])
display(data.head())

## 2. Сначала посмотреть, потом моделировать

Прежде чем выбирать, каким быть классификатору, выясните, что ему придётся переварить. Вопросы ниже
— именно те, которые окажутся важны для этой модели. Ответьте на них в четырёх клетках ниже, по
одной на блок, и сохраните числа: следующие разделы будут к ним обращаться.

**A. Размер и типы.**

1. `data.head()` и `data.dtypes`: какие столбцы пришли числами, а какие строками? Сопоставьте это со
   столбцом `type` таблицы `variables` — всюду ли pandas и UCI согласны?
2. `data.nunique()`, отсортированный, прочитанный как лестница. Где «горстка меток» превращается в
   «почти по значению на строку»? Так ли чиста эта ступенька, как вы ожидали, и застряло ли что-то
   посередине?

**B. Целевая переменная и пропуски.**

3. `data[TARGET].value_counts()`. Различных строк-меток больше двух — выясните, сколько их и почему.
   (Adult поставляется двумя файлами, и в одном из них ставят точку.) Получив такой столбец
   необработанным, классификатор решал бы другую задачу; какую именно? Исправьте столбец на месте —
   блок D читает его обратно. Раздел 4 повторяет это исправление для основного пути, так что ничего
   дальше не зависит от того, сделали ли вы его здесь.
4. После исправления: какая доля строк принадлежит положительному классу? Модель, всегда отвечающая
   большинством, набирает ровно дополнение к этой доле. Запишите это число — ниже его придётся
   превзойти.
5. `data.isna().sum()`, затем число строк хотя бы с одним пропуском. Все они приходятся на три
   столбца. Выразите потерю в процентах, прежде чем решать, дёшево ли выбросить эти строки.

**C. Числовые столбцы.**

6. `data.describe()`. `fnlwgt` — вес выборки переписи, посмотрите на его диапазон. `capital-gain` —
   какая доля строк *в точности* равна нулю? Постройте гистограмму каждого; ни один не похож на ту
   колокольную кривую, которую хотелось бы увидеть гауссиане.
7. Теперь постройте рядом `age` и `education-num`. Один из них — измерение, другой — лестница
   меток, которую просто записали целыми числами. Кто есть кто, и что в графике это выдаёт?
   `pd.crosstab(data["education"], data["education-num"])` закрывает вопрос окончательно.

**D. Есть ли в этом сигнал?**

8. Возьмите один категориальный столбец — `relationship` подойдёт — и посчитайте долю
   положительного класса по категориям, сравнивая с базовой долей из вопроса 4. Какие категории
   сдвигают её сильнее всего и насколько?
9. Для `age` и для `fnlwgt` посмотрите на распределение при условии метки:
   `data.groupby(TARGET)["age"].describe()` или две наложенные гистограммы. Один из двух разделяет
   классы, другой не разделяет их вообще. Кто есть кто?
10. Наконец, вопрос про *пары* столбцов. `education` и `education-num` сообщают вам две вещи или
    одну дважды? А `marital-status` и `relationship`? Запомните ответ — раздел 6 вернётся к нему и
    даст ему имя.

In [ ]:
# A — размер и типы.
# 1. `data.head()`, `data.dtypes` — против `variables["type"]`.
# 2. `data.nunique().sort_values()` — где ступенька и что застряло посередине?

# YOUR CODE HERE

In [ ]:
# B — целевая переменная и пропуски.
# 3. Сколько различных строк-меток в `data[TARGET]` и почему больше двух?
# 4. Доля положительного класса после чистки и порог «отвечать большинством».
# 5. `data.isna().sum()`, число строк хотя бы с одним пропуском и это же в процентах.

# YOUR CODE HERE

In [ ]:
# C — числовые столбцы.
# 6. `data.describe()`; диапазон `fnlwgt`; доля нулей в `capital-gain`; гистограмма каждого.
# 7. `age` и `education-num` рядом, плюс
#    `pd.crosstab(data["education"], data["education-num"])`.

# YOUR CODE HERE

In [ ]:
# D — сигнал и перекрытие.
# 8. Доля положительного класса по категориям `relationship`, против базовой доли.
# 9. `age` и `fnlwgt` при условии метки — который из них разделяет классы?
# 10. `education` против `education-num` и `marital-status` против `relationship`: две вещи или одна?

# YOUR CODE HERE

### Чем данные оказываются на самом деле

Сверьте свои ответы с этими — дальше ноутбук считает их известными.

- **Метку пришлось чистить.** Adult поставляется обучающим файлом и тестовым, и в тестовом класс
  записан с точкой на конце, так что в исходном столбце четыре строки на два класса: `<=50K`
  (24 720), `<=50K.` (12 435), `>50K` (7 841), `>50K.` (3 846). После удаления точки положительных
  строк 24.2%, значит порог «отвечать большинством» — **75.8%** accuracy.
- **Пропуски сосредоточены, и выбросить их дёшево.** Все они приходятся на `occupation` (966),
  `workclass` (963) и `native-country` (274). Хотя бы один пропуск есть в 1 221 строке — 2.5%, —
  остаётся 47 621. Контест именно их и выбрасывает.
- **Ни одна лестница не раскладывает столбцы по видам.** Число уникальных значений идёт 2 (`sex`),
  5, 6, 7, 9, 15, 16, 16, 42 (`native-country`), 74 (`age`), 96, 99, 123, 28 523 (`fnlwgt`) — без
  чистого разрыва: 42 страны стоят прямо рядом с 74 возрастами. Тип данных решает дело не лучше:
  `education-num` — это `int64`, и `capital-gain` тоже `int64`, а это совершенно разные по сути
  вещи.
- **`education-num` — это `education`, пронумерованный.** Таблица сопряжённости — матрица
  перестановки 16 × 16, то есть биекция. Это одна переменная, записанная дважды: один раз меткой,
  другой — номером ступени.
- **Два числовых столбца ведут себя плохо.** `fnlwgt` — вес выборки от 12 285 до 1 490 400, и его
  условные распределения по классам почти совпадают (средние 190 186 и 188 292): про доход он
  говорит почти ничего. `capital-gain` — 91.7% точных нулей с длинным тонким хвостом, и тот сигнал,
  который в нём есть, в основном про то, *равен* ли он нулю (95.8% строк класса `<=50K` против
  78.7% строк `>50K`), а такой формы ни у одной колокольной кривой нет. Оба вернутся в разделе 9.
- **Сигнала в данных хватает.** Средний возраст — 36.9 в классе `<=50K` против 44.2 в `>50K`;
  среднее число часов в неделю — 39.0 против 45.5. `relationship` разгоняет долю положительного
  класса от 1.5% (`Own-child`) до 47.4% (`Wife`) вокруг базовых 24.2%. Это обучаемая задача, а не
  бросок монеты.

## 3. Два вида признаков и что каждый из них допускает

Раздел 2 на глаз разложил четырнадцать столбцов на две кучки. Стоит точно сказать, что это за
кучки, потому что дальше модель выводится из этого различия почти механически.

### Категориальный признак

`workclass` принимает одну из девяти меток — `Private`, `Self-emp-not-inc`, `Federal-gov` и так
далее. Множество конечно, порядка у меток нет, и арифметика на них ничего не значит: `Private` — не
вдвое `State-gov` и не лежит между двумя другими категориями. Поэтому единственное, что обучающие
строки могут сказать про такой столбец, — как часто встречалась каждая метка, а единственный
вопрос, который ему можно задать на предсказании, — *как часто эта метка встречалась в этом
классе*. Значит, оценка — это **таблица**: по одному числу на пару (класс, категория), шириной в
$K_j$ категорий.

С таблицей идут два неудобных следствия, и оба проявятся в коде, который вы напишете. Клетка может
оказаться пустой — какая-то метка никогда не встретилась вместе с каким-то классом, — а пустая
клетка означает нулевую вероятность, которая внутри произведения обнуляет мнение всех остальных
признаков. И на предсказании может прийти метка, для которой в таблице вообще нет столбца; её всё
равно надо как-то оценить.

### Непрерывный признак

`age` — это измерение: упорядоченное, с осмысленными разностями, и на 47 621 строке оно принимает 74
различных значения. `fnlwgt` принимает 27 946. Здесь таблица не просто неудобна — это неправильный
объект. Каждая её клетка опиралась бы на горстку строк, а значение, которого не было в обучении, —
случай обычный, а не редкая случайность, так что таблица отвечала бы «вероятность нуль» на
большую часть теста.

На её место приходит **плотность**: одна функция на всей числовой прямой, подогнанная по значениям
внутри класса и вычислимая в точках, которых она не видела. Нормальное распределение со средним и
дисперсией самого класса — самая дешёвая такая функция и здесь значение по умолчанию. Ядерная оценка
плотности — очевидный следующий шаг, и раздел 9 про то, когда этот шаг себя оправдывает.

### И третья сущность, переодетая во вторую

`education-num` хранится как `int64`, так что любая проверка типа в pandas относит его к числам.
Ваша таблица сопряжённости показала, что это на самом деле: номер ступени для `education`,
шестнадцать уровней, в биекции со строковым столбцом, стоящим парой позиций левее. Арифметика на
нём *почти* осмысленна — ступени действительно упорядочены, и доля положительного класса в целом
растёт вдоль них, от 1.3% у `Preschool` до 74.4% у `Prof-school`, — но расстояние между ступенями
выдумано. Одна ступень от `Bachelors` к `Masters` — это не то же количество чего бы то ни было, что
одна ступень от `9th` к `10th`.

Это как раз тот случай, который тип хранения решить не может, и поэтому следующий раздел читает типы
из объявленных метаданных, а не угадывает их по значениям.

### Почему эта модель

У классификатора, который настаивает на одном виде признаков, есть два выхода, и оба плохи. Оставить
числа и выбросить метки — тогда `occupation`, `education` и `marital-status`, три сильнейших столбца
таблицы, идут в корзину. Или загнать метки в числа — а это утверждение о порядке и расстояниях,
которых в данных нет, и заодно заявка гауссиане, что `Private` = 3 лежит ровно посередине между
категориями 2 и 4.

Наивному Байесу не нужно ни то, ни другое, и ровно поэтому задание — про эту модель. Его
единственное предположение — признаки независимы, когда класс известен, — превращает правдоподобие
в **произведение по одному множителю на признак**, и нигде не требуется, чтобы множители были из
одного семейства. Таблица на девять клеток для `workclass`, нормальная плотность для `age`, ядерная
оценка для `capital-gain`: перемножьте их, и смешанная запись обрабатывается как есть, без
кодирования и без разбиения на корзины. Раздел 6 выписывает это произведение по слагаемым.

## 4. Как узнать вид столбца из метаданных

Контесту нужно правило, а не решение по обстоятельствам, поэтому он следует объявленному типу UCI с
одной поправкой: переменная, которую UCI называет `Integer`, но которая принимает не больше десяти
различных значений, — это код, а не измерение. Это обобщение случая `education-num` из прошлого
раздела.

На Adult поправка не срабатывает. У `education-num` шестнадцать уровней, это выше порога, так что
контест отдаёт его гауссиане, хотя мы с вами знаем, что это лестница — цена правила, которое должно
работать без присмотра, и цена небольшая, потому что ступени хотя бы упорядочены. Правило
отрабатывает своё на другом столбце — на том, где четырёхуровневая шкала записана как `1..4`:
UCI объявляет его `Integer`, по сути это код, и в роли измерения он тихо катастрофичен.

Строки с любым пропуском выбрасываются. Классификатор, который вы пишете, не обязан обрабатывать
`NaN`, а тихое заполнение пропусков изменило бы те числа, на которых мерили эталон.

In [ ]:
MAX_DISCRETE_CARDINALITY = 10

categorical_names, continuous_names = [], []
for column in adult.data.features.columns:
    declared = str(variables.loc[column, "type"])
    coded = declared == "Integer" and data[column].nunique() <= MAX_DISCRETE_CARDINALITY
    if declared in {"Continuous", "Integer"} and not coded:
        continuous_names.append(str(column))
    else:
        categorical_names.append(str(column))

clean = data.loc[data.notna().all(axis=1)].copy()
# В тестовой половине Adult метки записаны с точкой на конце; приводим оба вида к одному.
clean[TARGET] = clean[TARGET].astype(str).str.strip().str.rstrip(".")
features = clean[categorical_names + continuous_names]
target = clean[TARGET]

print(f"категориальные ({len(categorical_names)}): {categorical_names}")
print(f"непрерывные    ({len(continuous_names)}): {continuous_names}")
print(f"оставлено {len(clean)} строк из {len(data)}")
print(target.value_counts().to_string())

## 5. Разбиение, которое не съезжает

`train_test_split` берёт случайное разбиение. Поменяйте seed, порядок строк или версию pandas — и
разбиение поменяется вместе с ними, а с ним и любая измеренная на нём метрика, из-за чего эталонное
число теряет смысл. Поэтому контест разбивает по хешу **содержимого самой строки**:

$$\text{строка} \longmapsto \texttt{blake2b}(\text{строка}) \bmod 10^4 < 2500 \implies \text{тест}.$$

Отсюда следуют три свойства, и все три оправдывают лишний код. Разбиение — чистая функция от данных,
поэтому оно одинаково на любой машине и при любом запуске. Одинаковые строки всегда попадают на одну
сторону, так что повторяющаяся запись не может протечь из обучения в тест. А добавление строк в
датасет не меняет уже сделанного распределения.

Баланс классов сохраняется только приблизительно — это не стратифицированное разбиение, — и это
единственная заплаченная цена.

In [ ]:
categorical_values, continuous_values = mixed_nb_data.normalize_frame(
    features, categorical_names, continuous_names
)
labels = target.to_numpy(dtype=np.str_)

is_test = mixed_nb_data.hash_split(
    categorical_values, continuous_values, labels, test_fraction=0.25
)
is_train = ~is_test

print(f"обучение {is_train.sum()} / тест {is_test.sum()} ({is_test.mean():.3%} в тестовой части)")
print(
    "повторный запуск ничего не меняет:",
    np.array_equal(
        is_test,
        mixed_nb_data.hash_split(categorical_values, continuous_values, labels, 0.25),
    ),
)
print(
    "доля положительного класса, обучение против теста:",
    f"{(labels[is_train] == '>50K').mean():.4f}",
    f"{(labels[is_test] == '>50K').mean():.4f}",
)

In [ ]:
def build_matrix(regime: str) -> tuple[np.ndarray, list[int]]:
    """Возвращает object-матрицу признаков и индексы категориальных столбцов для одного режима."""
    blocks = {
        "categorical": [categorical_values],
        "continuous": [continuous_values],
        "mixed": [categorical_values, continuous_values],
    }[regime]
    matrix = np.concatenate([block.astype(object) for block in blocks], axis=1)
    n_categorical = 0 if regime == "continuous" else len(categorical_names)
    return matrix, list(range(n_categorical))


def split(
    regime: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[int]]:
    """Возвращает `(X_train, X_test, y_train, y_test, categorical_indices)` для одного режима."""
    matrix, categorical_indices = build_matrix(regime)
    return (
        matrix[is_train],
        matrix[is_test],
        labels[is_train],
        labels[is_test],
        categorical_indices,
    )


for regime in mixed_nb_data.REGIMES:
    X_train, X_test, _, _, categorical_indices = split(regime)
    print(
        f"{regime:11s} обучение {X_train.shape}  тест {X_test.shape}  "
        f"категориальных столбцов {len(categorical_indices)}"
    )

## 6. Модель по слагаемым

То, что вы сейчас реализуете, — это правило Байеса с одним прикрученным к нему предположением. Для
записи $x = (x_1, \dots, x_d)$ и класса $c$

$$p(y = c \mid x) = \frac{p(y = c)\, p(x \mid y = c)}{p(x)},$$

а *наивное* предположение состоит в том, что **когда класс известен**, признаки независимы:

$$p(x \mid y = c) = \prod_{j=1}^{d} p(x_j \mid y = c).$$

Возьмите логарифм — произведение четырнадцати правдоподобий обнулится в double задолго до конца — и
разбейте сумму по видам признаков:

$$\log p(y = c \mid x) = \underbrace{\log \pi_c}_{\text{априорное}}
    + \underbrace{\sum_{j \in C} \log \hat p_{cj}(x_j)}_{\text{категориальные}}
    + \underbrace{\sum_{j \in Q} \log \hat f_{cj}(x_j)}_{\text{непрерывные}}
    - \underbrace{\log Z(x)}_{\text{нормировка}},$$

где $C$ — категориальные признаки, $Q$ — непрерывные. Четыре слагаемых, по одному за раз.

### $\log \pi_c$ — априорное распределение

Насколько класс част до того, как запись прочитана; оценивается частотой в обучении,
$\hat\pi_c = N_c / N$. На Adult это $0.758$ и $0.242$, то есть одно априорное распределение уже
отвечает `<=50K` на всё и оказывается правым в трёх случаях из четырёх — тот самый порог из вопроса
4. Задача всех остальных слагаемых — сдвинуть запись с этого ответа по умолчанию. Параметр `priors`
перебивает частоты: передайте `[0.5, 0.5]` и вы спрашиваете, что говорит сама запись, не обращая
внимания на то, насколько класс част, — а это именно то, что нужно, когда баланс в обучении есть
артефакт сбора данных, а не факт о мире. `self._class_priors` уже разбирается с этим за вас.

### $\log \hat p_{cj}(x_j)$ — по множителю на категориальный столбец

Посчитать и нормировать внутри класса:

$$\hat p_{cj}(a) = \frac{N_{cja} + \alpha}{N_c + \alpha K_j},$$

где $N_{cja}$ — число обучающих строк класса $c$, у которых признак $j$ равен $a$, $N_c$ — число
строк класса $c$, а $K_j$ — число различных категорий признака $j$, встреченных при обучении. При
$\alpha = 0$ это обычная условная частота, и именно её использовать нельзя: одна пустая клетка —
категория, ни разу не встретившаяся вместе с классом $c$ в обучении, — обнуляет всё произведение,
то есть даёт $-\infty$ в логарифмах, и никакие свидетельства остальных тринадцати столбцов его уже
не вытащат. Прибавление $\alpha$ к каждому счётчику подкладывает дно под каждую клетку ценой
небольшого сдвига к равномерности; $\alpha K_j$ в знаменателе — то, что держит сумму по строке
равной единице. Это сглаживание Лидстоуна, а $\alpha = 1$ — частный случай, за который выступал
Лаплас.

У категории, впервые встреченной **на предсказании**, в таблице нет вообще никакого столбца.
Оценивайте её как случай с нулевым счётчиком,

$$\hat p_{cj}(\text{невстреченная}) = \frac{\alpha}{N_c + \alpha K_j},$$

и это то, что хранит `unseen_log_prob_`. Обратите внимание: значение одинаково для любой
невстреченной категории и меняется между классами только через $N_c$, — так что запись, у которой
все категории новые, откатывается почти ровно к априорному распределению, а это и есть правильное
высказывание о записи, про которую вы ничего не знаете.

### $\log \hat f_{cj}(x_j)$ — по множителю на непрерывный столбец

Таблица здесь безнадёжна по причинам из раздела 3, поэтому оценивайте **плотность** $\hat f_{cj}$:
функцию на всей прямой, подогнанную по значениям признака $j$ среди обучающих строк класса $c$ и
только их. Два свойства плотностей сбивают с толку, и оба всплывут при отладке:

- Плотность — не вероятность. $\hat f(x)$ может быть больше 1, а для узкого пика — намного больше,
  так что логарифм плотности выше нуля нормален, а не признак ошибки.
- В формуле *не хватает* того $\mathrm{d}x$, который превратил бы её в вероятность. Этот множитель
  одинаков для всех классов и потому сокращается при нормировке ниже — именно поэтому складывать в
  одной сумме логарифм плотности и логарифм вероятности законно. Апостериорное распределение
  выходит верным; сам же совместный скор верен только с точностью до этого общего множителя.

По умолчанию $\hat f$ — это `GaussianDensity`, нормальное распределение, обученное методом
максимального правдоподобия, так что $\hat\mu_{cj}$ и $\hat\sigma^2_{cj}$ — просто среднее и
дисперсия одного столбца внутри одного класса, а

$$\log \hat f_{cj}(x) = -\tfrac{1}{2}\left[
    \log\!\left(2\pi\hat\sigma^2_{cj}\right) + \frac{(x - \hat\mu_{cj})^2}{\hat\sigma^2_{cj}}
\right].$$

Дисперсия ограничена снизу параметром `var_smoothing`, потому что у столбца, оказавшегося постоянным
внутри класса, $\hat\sigma^2 = 0$, и выражение стало бы бесконечным во всех остальных точках. При
этом к гауссиане вас никто не привязывает: `density_estimator` принимает любой объект с парой
`fit` / `score_samples`, как у `KernelDensity`, — а после того, как раздел 2 показал вам
`capital-gain`, вы и сами догадываетесь, зачем это нужно.

### $-\log Z(x)$ — нормировка

Нормирующий множитель, $Z(x) = p(x) = \sum_{c'} \pi_{c'} \prod_j p(x_j \mid y = c')$, сумма по
классам. От $c$ он не зависит, поэтому сдвигает скоры всех классов на одну и ту же величину и не
меняет ни один arg-max — `predict` мог бы его вообще не считать. Он нужен затем, чтобы скоры стали
распределением с суммой единица, а это требуется `predict_proba` и `log_loss`. В логарифмах это
`logsumexp` по оси классов — и **это слагаемое уже написано**: `predict_log_proba` вычитает его из
того, что вернул `_joint_log_likelihood`. Ваша работа заканчивается на первых трёх слагаемых.

### Насколько «наивное» неверно?

Здесь — сильно неверно, и об этом полезно знать до того, как числа вас удивят. Вопрос 10 показал,
что `education` и `education-num` — одна переменная дважды, а `marital-status` и `relationship`
перекрываются почти полностью (`Married-civ-spouse` — это `Husband` или `Wife` во всех строках,
кроме нескольких сотен). Свидетельство каждой такой пары учитывается дважды, и апостериорные
вероятности выходят слишком уверенными, прижатыми к 0 и 1 сильнее, чем позволяют данные. Arg-max это
в основном выдерживает — поэтому наивный Байес остаётся приличным классификатором, будучи плохой
оценкой вероятностей, и поэтому контест оценивает accuracy и macro $F_1$, а не калибровку.

## 7. Что реализуете вы

Четыре пропуска в `naive_bayes.py`, в порядке возрастания сложности:

1. `GaussianDensity.fit` — $\hat\mu$ и $\hat\sigma^2$ методом максимального правдоподобия,
   дисперсия ограничена снизу через `var_smoothing`.
2. `GaussianDensity.score_samples` — логарифм плотности, выписанный в третьем слагаемом выше,
   вычисленный в каждой запрошенной точке.
3. `MixedNaiveBayes.fit` — априорные вероятности, сглаженные категориальные таблицы и по одному
   клонированному оценщику плотности на класс и на непрерывный признак.
4. `MixedNaiveBayes._joint_log_likelihood` — $\log \pi_c$ плюс по слагаемому на признак. Четвёртое
   слагаемое не ваше.

Две ловушки, о которых стоит сказать заранее. Работайте в логарифмах везде: произведение
четырнадцати правдоподобий обнуляется в double, а `predict_proba` выглядит прилично только потому,
что `predict_log_proba` нормирует через `logsumexp`. И клонируйте оценщик плотности через
`sklearn.base.clone` перед обучением — вызывающая сторона передаёт **один** объект на все классы и
признаки, и обучение на месте оставило бы все классы с последней подгонкой.

Всё, что не является моделью — работа со столбцами, нормировка апостериорного распределения,
валидация параметров, — уже написано, а клетка ниже перезагружает модуль, так что файл можно править
и запускать заново без перезапуска ядра.

In [ ]:
import naive_bayes

importlib.reload(naive_bayes)
GaussianDensity = naive_bayes.GaussianDensity
MixedNaiveBayes = naive_bayes.MixedNaiveBayes

X_train, X_test, y_train, y_test, categorical_indices = split("mixed")
model = MixedNaiveBayes(categorical_features=categorical_indices).fit(X_train, y_train)
predicted = model.predict(X_test)

print(f"классы             {model.classes_.tolist()}")
print(f"априорные          {np.exp(model.class_log_prior_)}")
print(
    f"категор. / непрер. {model.categorical_features_.tolist()} / "
    f"{model.continuous_features_.tolist()}"
)
print(f"accuracy           {accuracy_score(y_test, predicted):.4f}")
print(f"macro F1           {f1_score(y_test, predicted, average='macro'):.4f}")
log_loss_value = log_loss(y_test, model.predict_proba(X_test), labels=model.classes_)
print(f"log loss           {log_loss_value:.4f}")

### Эталонные числа

Что получает эталонная реализация на этом датасете со значениями по умолчанию — `alpha=1.0`,
`GaussianDensity`, хеш-разбиение 75/25, 35 713 обучающих и 11 908 тестовых строк после выбрасывания
1 221 неполной строки:

| режим | accuracy | macro $F_1$ | log loss |
| --- | --- | --- | --- |
| только категориальные | 0.7956 | 0.7463 | 0.4573 |
| только непрерывные | 0.7995 | 0.6517 | 0.9065 |
| полная смешанная таблица | 0.8332 | 0.7448 | 0.6659 |

Попадите в примерно 0.02 от столбца accuracy — и ваша реализация согласуется с той, которая вас
оценивает. Оказаться заметно *выше* — плохая новость: почти всегда это значит, что тестовые строки
протекли в обучение.

Чего не стоит вычитывать из последней строки: смешанная таблица здесь выигрывает, но так быть не
обязано. Непрерывный столбец, который гауссиана описывает плохо — `capital-gain`, 91.7% нулей с
длинным хвостом, вы его уже видели, — может утянуть смесь *ниже* категориальной половины. Падение
при добавлении непрерывных столбцов — это результат, а не обязательно ошибка; лечится лучшей
плотностью, для чего и существует `density_estimator`.

## 8. Бейзлайны

Два из них — не просто бейзлайны, а эквивалентности, и это самый острый отладочный инструмент из
имеющихся:

- на одних непрерывных признаках со значением по умолчанию `GaussianDensity` модель **совпадает** с
  `GaussianNB(var_smoothing=0)` — апостериорные вероятности должны согласоваться до машинной
  точности;
- на одних категориальных она **совпадает** с `CategoricalNB(alpha=1)` на столбцах, закодированных
  порядковым кодированием, — ровно так же.

Если расходится хоть одна, ошибка именно в этом правдоподобии, а это гораздо меньшее место для
поиска, чем «классификатор врёт».

In [ ]:
rows = []

X_train_c, X_test_c, y_train_c, y_test_c, _ = split("continuous")
X_train_c = X_train_c.astype(np.float64)
X_test_c = X_test_c.astype(np.float64)
mine = MixedNaiveBayes(categorical_features=[]).fit(X_train_c, y_train_c)
theirs = GaussianNB(var_smoothing=0.0).fit(X_train_c, y_train_c)
rows.append(
    (
        "непрерывные: моя против GaussianNB",
        accuracy_score(y_test_c, mine.predict(X_test_c)),
        accuracy_score(y_test_c, theirs.predict(X_test_c)),
        np.abs(mine.predict_log_proba(X_test_c) - theirs.predict_log_proba(X_test_c)).max(),
    )
)

X_train_k, X_test_k, y_train_k, y_test_k, categorical_k = split("categorical")
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(X_train_k)
mine_k = MixedNaiveBayes(categorical_features=categorical_k, alpha=1.0).fit(X_train_k, y_train_k)
theirs_k = CategoricalNB(alpha=1.0, force_alpha=True).fit(encoder.transform(X_train_k), y_train_k)
rows.append(
    (
        "категориальные: моя против CategoricalNB",
        accuracy_score(y_test_k, mine_k.predict(X_test_k)),
        accuracy_score(y_test_k, theirs_k.predict(encoder.transform(X_test_k))),
        np.abs(
            mine_k.predict_log_proba(X_test_k)
            - theirs_k.predict_log_proba(encoder.transform(X_test_k))
        ).max(),
    )
)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
rows.append(
    (
        "смешанная: моя против большинства",
        accuracy_score(y_test, model.predict(X_test)),
        accuracy_score(y_test, dummy.predict(X_test)),
        float("nan"),
    )
)

display(pd.DataFrame(rows, columns=["сравнение", "моя", "эталон", "max |Δ log p|"]))

## 9. Замена непрерывного правдоподобия

`density_estimator` — та часть задания, которая уже не про наивного Байеса: непрерывный признак
может описать что угодно с парой `fit` / `score_samples`, как у `KernelDensity`, а гауссиана стоит
по умолчанию лишь потому, что это самый дешёвый честный выбор.

Два обстоятельства, о которых стоит помнить, когда берётесь за настоящую KDE.

**Масштаб.** `KernelDensity(bandwidth="scott")` ставит $h = n^{-1/(d+4)}$ и *не* умножает на
стандартное отклонение признака. На `capital-gain`, который доходит до $10^5$, ширина окна $0.25$
превращает плотность в гребёнку из пиков. Либо сначала стандартизуйте непрерывный блок, либо
передавайте ширину окна в единицах самого признака.

**Стоимость.** Подсчёт KDE — это $O(n_\text{train} \cdot n_\text{test})$ на класс и на признак
против $O(n_\text{train} + n_\text{test})$ у гауссианы. На полном Adult это миллиарды вычислений
ядра, поэтому сравнение ниже идёт на детерминированной подвыборке — вырезанной тем же хеш-приёмом,
но с другой солью, так что и она воспроизводима.

In [ ]:
subsample = mixed_nb_data.hash_split(
    categorical_values,
    continuous_values,
    labels,
    test_fraction=0.03,
    salt="kde-demo",
)
sub_train, sub_test = subsample & is_train, subsample & is_test
scaler = StandardScaler().fit(continuous_values[sub_train])
X_sub_train = scaler.transform(continuous_values[sub_train])
X_sub_test = scaler.transform(continuous_values[sub_test])
y_sub_train, y_sub_test = labels[sub_train], labels[sub_test]
print(f"подвыборка: {len(y_sub_train)} обучение / {len(y_sub_test)} тест")

rows = []
candidates = {
    "GaussianDensity (по умолчанию)": None,
    "KDE gaussian h=0.2": KernelDensity(kernel="gaussian", bandwidth=0.2),
    "KDE gaussian h=0.5": KernelDensity(kernel="gaussian", bandwidth=0.5),
    "KDE epanechnikov h=0.5": KernelDensity(kernel="epanechnikov", bandwidth=0.5),
    "KDE tophat h=0.8": KernelDensity(kernel="tophat", bandwidth=0.8),
}
for name, estimator in candidates.items():
    started = time.perf_counter()
    candidate = MixedNaiveBayes(categorical_features=[], density_estimator=estimator).fit(
        X_sub_train, y_sub_train
    )
    fitted_at = time.perf_counter()
    guess = candidate.predict(X_sub_test)
    rows.append(
        (
            name,
            accuracy_score(y_sub_test, guess),
            f1_score(y_sub_test, guess, average="macro"),
            (fitted_at - started) * 1e3,
            (time.perf_counter() - fitted_at) * 1e3,
        )
    )

display(pd.DataFrame(rows, columns=["плотность", "accuracy", "macro F1", "fit, мс", "predict, мс"]))

## 10. Куда уходит время

Контест измеряет три стадии — создание оценщика, обучение, предсказание — и сравнивает каждую со
временем эталонного решения, применяя широкий множитель плюс нижний порог. Широкий настолько, чтобы
хватило медленной машине; и не настолько, чтобы хватило питоновскому циклу по строкам, который
переобучает плотность внутри `predict`. `mixed_nb_eval.evaluate` — ровно та функция, которой
пользуется проверяющая система, так что эта таблица — та самая, которую она и построит.

In [ ]:
timings = []
for regime in mixed_nb_data.REGIMES:
    X_tr, X_te, y_tr, y_te, categorical_indices = split(regime)
    evaluation = mixed_nb_eval.evaluate(
        lambda indices=categorical_indices: MixedNaiveBayes(categorical_features=indices),
        X_tr,
        y_tr,
        X_te,
        y_te,
    )
    timings.append(
        {
            "режим": regime,
            "accuracy": evaluation.accuracy,
            "macro F1": evaluation.macro_f1,
            "log loss": evaluation.log_loss,
            **{
                f"{stage}, мс": evaluation.runtime_seconds[stage] * 1e3
                for stage in mixed_nb_eval.STAGES
            },
        }
    )

display(pd.DataFrame(timings).set_index("режим"))

## 11. Локальные тесты

Шесть проверок, каждая за одну группу тестов проверяющей системы. Они идут на этом датасете, а не на
том, который у системы, поэтому пройти их все необходимо, но недостаточно, — зато каждая из них
падает по конкретной, называемой причине, и именно поэтому их стоит запустить перед отправкой.

In [ ]:
# Тест 1 — контракт оценщика sklearn.
from sklearn.base import clone
from sklearn.exceptions import NotFittedError
from sklearn.pipeline import Pipeline

fresh = MixedNaiveBayes(categorical_features=categorical_indices)
assert set(fresh.get_params()) == {
    "categorical_features",
    "density_estimator",
    "alpha",
    "priors",
}, "сигнатуру конструктора менять нельзя"
assert clone(fresh).get_params()["alpha"] == 1.0, "clone должен сохранять параметры"
try:
    clone(fresh).predict(X_test)
except NotFittedError:
    pass
else:
    raise AssertionError("predict до fit должен выбрасывать NotFittedError")

X_tr, X_te, y_tr, y_te, indices = split("mixed")
assert MixedNaiveBayes(categorical_features=indices).fit(X_tr, y_tr) is not None
scores = cross_val_score(
    MixedNaiveBayes(categorical_features=indices), X_tr[:3000], y_tr[:3000], cv=4
)
assert np.all(np.isfinite(scores)), f"cross_val_score вернул {scores}"
assert (
    Pipeline([("nb", MixedNaiveBayes(categorical_features=indices))])
    .fit(X_tr, y_tr)
    .score(X_te, y_te)
    > 0.6
), "оценщик должен работать как шаг Pipeline"
print(f"OK — контракт соблюдён, accuracy на 4-fold CV {scores.mean():.4f}")

In [ ]:
# Тест 2 — только категориальные признаки: модель совпадает с CategoricalNB.
X_tr, X_te, y_tr, y_te, indices = split("categorical")
mine = MixedNaiveBayes(categorical_features=indices, alpha=1.0).fit(X_tr, y_tr)
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(X_tr)
theirs = CategoricalNB(alpha=1.0, force_alpha=True).fit(encoder.transform(X_tr), y_tr)
gap = np.abs(mine.predict_log_proba(X_te) - theirs.predict_log_proba(encoder.transform(X_te))).max()
assert gap < 1e-6, f"апостериорные расходятся с CategoricalNB на {gap:.3e}"
print(f"OK — совпадает с CategoricalNB до {gap:.2e}; accuracy {mine.score(X_te, y_te):.4f}")

In [ ]:
# Тест 3 — только непрерывные признаки: модель совпадает с GaussianNB.
X_tr, X_te, y_tr, y_te, _ = split("continuous")
X_tr, X_te = X_tr.astype(np.float64), X_te.astype(np.float64)
# У `fnlwgt` в Adult дисперсия порядка 1e10, поэтому два логарифма плотности сокращаются до
# примерно 1e-9 по модулю даже когда оба верны; 1e-6 всё равно строже любой ошибки.
mine = MixedNaiveBayes(categorical_features=[]).fit(X_tr, y_tr)
theirs = GaussianNB(var_smoothing=0.0).fit(X_tr, y_tr)
gap = np.abs(mine.predict_log_proba(X_te) - theirs.predict_log_proba(X_te)).max()
assert gap < 1e-6, f"апостериорные расходятся с GaussianNB на {gap:.3e}"
print(f"OK — совпадает с GaussianNB до {gap:.2e}; accuracy {mine.score(X_te, y_te):.4f}")

In [ ]:
# Тест 4 — смешанная таблица: вероятности согласованы, и смесь лучше любой из половин.
X_tr, X_te, y_tr, y_te, indices = split("mixed")
mixed = MixedNaiveBayes(categorical_features=indices).fit(X_tr, y_tr)
report = mixed_nb_eval.probability_report(mixed, X_te)
assert report["row_sum_deviation"] < 1e-9, "строки predict_proba должны суммироваться в единицу"
assert report["log_gap"] < 1e-9, "predict_log_proba должен быть log(predict_proba)"
assert report["argmax_disagreement"] == 0.0, "predict должен быть arg-max от predict_proba"

mixed_accuracy = mixed.score(X_te, y_te)
majority = (y_te == pd.Series(y_tr).mode()[0]).mean()
assert mixed_accuracy > majority, (
    f"accuracy на смешанной {mixed_accuracy:.4f} не превосходит большинство {majority:.4f}"
)
print(
    f"OK — вероятности согласованы; accuracy на смешанной {mixed_accuracy:.4f} против {majority:.4f}"
)

In [ ]:
# Тест 5 — переданный оценщик плотности используется и клонируется, а не обучается на месте.
from scipy.special import logsumexp
from sklearn.utils.validation import check_is_fitted

estimator = KernelDensity(kernel="gaussian", bandwidth=0.5)
model_kde = MixedNaiveBayes(categorical_features=[], density_estimator=estimator).fit(
    X_sub_train, y_sub_train
)
try:
    check_is_fitted(estimator)
except NotFittedError:
    pass
else:
    raise AssertionError("переданный оценщик должен клонироваться, а не обучаться на месте")

classes, encoded = np.unique(y_sub_train, return_inverse=True)
expected = np.tile(np.log(np.bincount(encoded) / encoded.shape[0]), (len(y_sub_test), 1))
for position in range(X_sub_train.shape[1]):
    for class_index in range(len(classes)):
        density = KernelDensity(kernel="gaussian", bandwidth=0.5).fit(
            X_sub_train[encoded == class_index, position].reshape(-1, 1)
        )
        expected[:, class_index] += density.score_samples(X_sub_test[:, position].reshape(-1, 1))
expected -= logsumexp(expected, axis=1, keepdims=True)
gap = np.abs(model_kde.predict_log_proba(X_sub_test) - expected).max()
assert gap < 1e-8, f"апостериорное с KDE расходится на {gap:.3e}"
print(f"OK — KernelDensity подключена верно, апостериорное совпадает до {gap:.2e}")

In [ ]:
# Тест 6 — сглаживание, невстреченные категории и заданные априорные вероятности.
alpha = 0.3
X_tr, X_te, y_tr, y_te, indices = split("categorical")
smoothed = MixedNaiveBayes(categorical_features=indices, alpha=alpha).fit(X_tr, y_tr)
_, encoded = np.unique(y_tr, return_inverse=True)
class_counts = np.bincount(encoded).astype(np.float64)

column = X_tr[:, 0].astype(str)
categories = smoothed.categories_[0]
assert list(categories) == list(np.unique(column)), (
    "categories_ должен содержать отсортированные категории"
)
for code_index, category in enumerate(categories):
    for class_index in range(len(class_counts)):
        joint = np.sum((column == category) & (encoded == class_index))
        expected_probability = (joint + alpha) / (
            class_counts[class_index] + alpha * len(categories)
        )
        assert np.isclose(
            np.exp(smoothed.feature_log_prob_[0][class_index, code_index]),
            expected_probability,
        ), f"сглаженная частота для {category!r} неверна"
assert np.allclose(
    np.exp(smoothed.unseen_log_prob_[:, 0]),
    alpha / (class_counts + alpha * len(categories)),
), "невстреченная категория должна оцениваться как категория с нулевым счётчиком"

novel = np.array([["never-seen"] * len(indices)], dtype=object)
expected_row = smoothed.class_log_prior_ + smoothed.unseen_log_prob_.sum(axis=1)
expected_row -= logsumexp(expected_row)
assert np.allclose(smoothed.predict_log_proba(novel)[0], expected_row), (
    "строка из невстреченных категорий должна оцениваться через unseen_log_prob_"
)

uniform = MixedNaiveBayes(categorical_features=indices, priors=[0.5, 0.5]).fit(X_tr, y_tr)
assert np.allclose(uniform.class_log_prior_, np.log(0.5)), "priors должен перебивать частоты"
print("OK — сглаживание Лидстоуна, невстреченные категории и заданные априорные работают")

## 12. Отправка

Загружайте **`naive_bayes.py`** — сам файл, не этот ноутбук и не архив. Проверяющая система
переименовывает его, импортирует из него `MixedNaiveBayes` и `GaussianDensity` и запускает свои
тесты; больше ничего в файле не исполняется, так что оставленные эксперименты в блоке `__main__`
безвредны, но бесполезны.

Два отличия от того, что вы здесь запускали, стоит ожидать заранее. Ваш классификатор оценивается на
датасетах, отличных от этого, так что ошибке, которой размер Adult и число его категорий давали где
спрятаться, деваться будет особо некуда, — потому же локальные проверки выше необходимы, но
недостаточны. И метрики сравниваются с эталонной реализацией двусторонне: оказаться далеко *выше*
эталона тоже означает, что вы обучаете не ту модель, которая описана.